In [5]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd

In [ ]:
# Caminhos de entrada
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "investimento_2016_2024.csv"

# Caminhos de saída
OUTPUT_DIR = DATA_DIR / "nip_pipeline_outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed"
REPORTS_DIR = OUTPUT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CLEAN_FILE = PROCESSED_DIR / "investimento_2016_2024_clean_step01.csv"

INPUT_FILE

WindowsPath('data/investimento_2016_2024.csv')

In [7]:
def normalize_column_name(col: str) -> str:
    """Padroniza nomes de colunas em snake_case simples."""
    col = str(col).strip().lower()
    col = unicodedata.normalize("NFKD", col)
    col = "".join(ch for ch in col if not unicodedata.combining(ch))
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def normalize_text_basic(value) -> str:
    """Normalização leve para preservar conteúdo original e contexto textual."""
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # remove espaços duplicados em cada linha, mas preserva quebras de linha
    text = "\n".join(re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n"))
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def create_text_base(title: str, body: str) -> str:
    title = normalize_text_basic(title)
    body = normalize_text_basic(body)
    if title and body:
        return f"{title}\n{body}"
    return title or body


def create_stable_id(row: pd.Series) -> str:
    """Cria ID estável a partir de campos que identificam a notícia.

    Usa SHA-256 e mantém os 24 primeiros caracteres para IDs menores.
    Não é usado para segurança; é apenas um identificador determinístico.
    """
    base = "|".join([
        str(row.get("data_publicacao", "")),
        normalize_text_basic(row.get("fonte", "")),
        normalize_text_basic(row.get("titulo", "")),
        normalize_text_basic(row.get("texto", ""))[:500],
    ])
    return hashlib.sha256(base.encode("utf-8")).hexdigest()[:24]

In [8]:
df_raw = pd.read_csv(INPUT_FILE)

print(f"Linhas: {df_raw.shape[0]:,}")
print(f"Colunas: {df_raw.shape[1]:,}")
display(df_raw.head(3))

Linhas: 15,287
Colunas: 5


,data_publicacao,titulo,fonte,texto,flagnoticia
0,2016-01-01 00:00:00,Araçoiaba busca melhorar abastecimento de água,Diário de Sorocaba,Para atender a demanda de crescimento dos bair...,I
1,2016-01-02 00:00:00,"Hospital da Criança,do Grendacc,será entregue ...",Jornal de Jundiaí,"Obras do Hospital da Criança, que vai atender ...",I
2,2016-01-02 00:00:00,Produtor investe R$ 100 mil em estufa,Cruzeiro do Sul,"No bairro Caguaçu, em Sorocaba, o produtor rur...",I


In [9]:
df = df_raw.copy()

# Padroniza nomes de colunas
df.columns = [normalize_column_name(c) for c in df.columns]

# Padroniza data
df["data_publicacao"] = pd.to_datetime(df["data_publicacao"], errors="coerce")

# Garante que flagnoticia seja string padronizada
df["flagnoticia"] = df["flagnoticia"].astype("string").str.strip().str.upper()

display(df.head(3))
print(df.dtypes)

,data_publicacao,titulo,fonte,texto,flagnoticia
0,2016-01-01,Araçoiaba busca melhorar abastecimento de água,Diário de Sorocaba,Para atender a demanda de crescimento dos bair...,I
1,2016-01-02,"Hospital da Criança,do Grendacc,será entregue ...",Jornal de Jundiaí,"Obras do Hospital da Criança, que vai atender ...",I
2,2016-01-02,Produtor investe R$ 100 mil em estufa,Cruzeiro do Sul,"No bairro Caguaçu, em Sorocaba, o produtor rur...",I


data_publicacao    datetime64[us]
titulo                        str
fonte                         str
texto                         str
flagnoticia                string
dtype: object


In [10]:
df["titulo_limpo"] = df["titulo"].apply(normalize_text_basic)
df["texto_limpo"] = df["texto"].apply(normalize_text_basic)
df["texto_base"] = df.apply(lambda row: create_text_base(row["titulo_limpo"], row["texto_limpo"]), axis=1)

df["titulo_len"] = df["titulo_limpo"].str.len()
df["texto_len"] = df["texto_limpo"].str.len()
df["texto_base_len"] = df["texto_base"].str.len()

display(df[["titulo", "titulo_limpo", "texto_base_len"]].head(3))

,titulo,titulo_limpo,texto_base_len
0,Araçoiaba busca melhorar abastecimento de água,Araçoiaba busca melhorar abastecimento de água,1627
1,"Hospital da Criança,do Grendacc,será entregue ...","Hospital da Criança,do Grendacc,será entregue ...",2448
2,Produtor investe R$ 100 mil em estufa,Produtor investe R$ 100 mil em estufa,1013


In [11]:
df["noticia_id"] = df.apply(create_stable_id, axis=1)

duplicated_ids = df["noticia_id"].duplicated().sum()
print(f"IDs duplicados: {duplicated_ids:,}")

display(df[["noticia_id", "data_publicacao", "fonte", "titulo_limpo"]].head(5))

IDs duplicados: 76


,noticia_id,data_publicacao,fonte,titulo_limpo
0,5c2f55d84fa33a4521c84952,2016-01-01,Diário de Sorocaba,Araçoiaba busca melhorar abastecimento de água
1,729d407367d6ce010c660b27,2016-01-02,Jornal de Jundiaí,"Hospital da Criança,do Grendacc,será entregue ..."
2,eb94b4766db90094be9dc437,2016-01-02,Cruzeiro do Sul,Produtor investe R$ 100 mil em estufa
3,265e1fc4bd247c649173c83d,2016-01-02,Folha da Região,Franquias da região conquistam mercado local e...
4,72519e6106b0f258b1880485,2016-01-02,Jornal de Jundiaí,"Hospital da Criança, do Grendac, será entregue..."


In [12]:
rows_before = len(df)
df_clean = df.drop_duplicates(subset=["noticia_id"], keep="first").copy()
rows_after = len(df_clean)

print(f"Registros antes da deduplicação: {rows_before:,}")
print(f"Registros após deduplicação: {rows_after:,}")
print(f"Removidos: {rows_before - rows_after:,}")

Registros antes da deduplicação: 15,287
Registros após deduplicação: 15,211
Removidos: 76


In [13]:
final_columns = [
    "noticia_id",
    "data_publicacao",
    "titulo",
    "titulo_limpo",
    "fonte",
    "texto",
    "texto_limpo",
    "texto_base",
    "flagnoticia",
    "titulo_len",
    "texto_len",
    "texto_base_len",
]

df_clean = df_clean[final_columns].sort_values(["data_publicacao", "noticia_id"]).reset_index(drop=True)

display(df_clean.head(3))
print(df_clean.shape)

,noticia_id,data_publicacao,titulo,titulo_limpo,fonte,texto,texto_limpo,texto_base,flagnoticia,titulo_len,texto_len,texto_base_len
0,5c2f55d84fa33a4521c84952,2016-01-01,Araçoiaba busca melhorar abastecimento de água,Araçoiaba busca melhorar abastecimento de água,Diário de Sorocaba,Para atender a demanda de crescimento dos bair...,Para atender a demanda de crescimento dos bair...,Araçoiaba busca melhorar abastecimento de água...,I,46,1580,1627
1,265e1fc4bd247c649173c83d,2016-01-02,Franquias da região conquistam mercado local e...,Franquias da região conquistam mercado local e...,Folha da Região,Quem caminha por ruas comerciais e pelos centr...,Quem caminha por ruas comerciais e pelos centr...,Franquias da região conquistam mercado local e...,I,60,3027,3088
2,72519e6106b0f258b1880485,2016-01-02,"Hospital da Criança, do Grendac, será entregue...","Hospital da Criança, do Grendac, será entregue...",Jornal de Jundiaí,"Obras do Hospital da Criança, que vai atender ...","Obras do Hospital da Criança, que vai atender ...","Hospital da Criança, do Grendac, será entregue...",I,66,2384,2451


(15211, 12)


In [14]:
df_clean.to_csv(OUTPUT_CLEAN_FILE, index=False, encoding="utf-8")

print(f"Arquivo salvo em: {OUTPUT_CLEAN_FILE}")
print(f"Linhas salvas: {len(df_clean):,}")

Arquivo salvo em: data\nip_pipeline_outputs\processed\investimento_2016_2024_clean_step01.csv
Linhas salvas: 15,211


In [15]:
summary = {
    "rows_clean": int(len(df_clean)),
    "unique_ids": int(df_clean["noticia_id"].nunique()),
    "date_min": str(df_clean["data_publicacao"].min()),
    "date_max": str(df_clean["data_publicacao"].max()),
    "nulls_final": df_clean.isna().sum().to_dict(),
    "output_clean_file": str(OUTPUT_CLEAN_FILE),
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

with open(REPORTS_DIR / "summary_clean_step01.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

,value
rows_clean,15211
unique_ids,15211
date_min,2016-01-01 00:00:00
date_max,2024-12-29 03:00:00
nulls_final,"{'noticia_id': 0, 'data_publicacao': 0, 'titul..."
output_clean_file,data\nip_pipeline_outputs\processed\investimen...
